# PromptSR evaluation notebook

这里补成可直接跑的 benchmark 评测入口：
- 配置 benchmark / checkpoint / device
- 加载训练好的模型权重
- 跑 Set5 / Set14 / BSD100 / Urban100 / Manga109
- 输出每张图指标、每个数据集均值，并可导出 CSV


In [4]:
from pathlib import Path
import sys

import pandas as pd
import torch

repo_root = Path('..').resolve()
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from promptsr.config import PromptSRConfig
from promptsr.eval import evaluate_benchmarks, paired_benchmark_paths
from promptsr.lit_module import PromptSRLightningModule
from promptsr.workflow import _load_compatible_state


In [5]:
benchmark_root = repo_root / 'data' / 'benchmarks'
available_datasets = sorted(p.name for p in benchmark_root.iterdir() if p.is_dir())
available_datasets


['BSD100', 'Manga109', 'Set14', 'Set5', 'Urban100']

## 1. 配置评测参数

默认指向 `outputs/notebook-run/checkpoints/last.ckpt`。
如果你要评别的权重，直接改 `checkpoint_path` 即可。


In [6]:
checkpoint_path = repo_root / 'outputs' / 'notebook-run' / 'checkpoints' / 'last.ckpt'
output_csv = repo_root / 'outputs' / 'benchmark_results' / 'promptsr_eval.csv'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = PromptSRConfig(
    scale=4,
    benchmark_root=str(benchmark_root),
    output_dir=str(repo_root / 'outputs' / 'benchmark_results'),
    init_checkpoint=str(checkpoint_path),
)

print('device =', device)
print('checkpoint exists =', checkpoint_path.exists(), checkpoint_path)
print('benchmark root exists =', benchmark_root.exists(), benchmark_root)


device = cpu
checkpoint exists = True /Users/dbydd/vibe-agent-working-dir/notes/groupshare/20260514/promptsr-reproduce/outputs/notebook-run/checkpoints/last.ckpt
benchmark root exists = True /Users/dbydd/vibe-agent-working-dir/notes/groupshare/20260514/promptsr-reproduce/data/benchmarks


## 2. 检查 benchmark 配对情况

这里先确认每个 benchmark 数据集都能正确匹配到 `HR` / `LR_bicubic/X4`。


In [7]:
pair_rows = []
for dataset_name in ['Set5', 'Set14', 'BSD100', 'Urban100', 'Manga109']:
    dataset_dir = benchmark_root / dataset_name
    pairs = paired_benchmark_paths(dataset_dir, cfg.scale)
    pair_rows.append({
        'dataset': dataset_name,
        'exists': dataset_dir.exists(),
        'num_pairs': len(pairs),
        'sample_lr': pairs[0][0].name if pairs else None,
        'sample_hr': pairs[0][1].name if pairs else None,
    })

pair_df = pd.DataFrame(pair_rows)
pair_df


,dataset,exists,num_pairs,sample_lr,sample_hr
0,Set5,True,5,babyx4.png,baby.png
1,Set14,True,14,baboonx4.png,baboon.png
2,BSD100,True,100,101085x4.png,101085.png
3,Urban100,True,100,img_001x4.png,img_001.png
4,Manga109,True,109,ARMSx4.png,ARMS.png


## 3. 加载模型与 checkpoint

这里复用训练流程里的 `_load_compatible_state`，这样 notebook 和训练代码的权重兼容策略保持一致。


In [8]:
model = PromptSRLightningModule(cfg)
if cfg.init_checkpoint:
    _load_compatible_state(model, cfg.init_checkpoint)
model = model.to(device)
model.eval()


loaded compatible init checkpoint /Users/dbydd/vibe-agent-working-dir/notes/groupshare/20260514/promptsr-reproduce/outputs/notebook-run/checkpoints/last.ckpt
loaded keys 590 skipped keys 0
missing keys 0 unexpected keys 0


PromptSRLightningModule(
  (model): PromptSRModel(
    (head): Conv2d(3, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (groups): ModuleList(
      (0-3): 4 x ResidualGroup(
        (blocks): ModuleList(
          (0-2): 3 x CascadePromptingBlock(
            (gapl): GlobalAnchorPromptingLayer(
              (q_proj): Conv2d(48, 48, kernel_size=(1, 1), stride=(1, 1))
              (k_proj): Conv2d(48, 48, kernel_size=(1, 1), stride=(1, 1))
              (v_proj): Conv2d(48, 48, kernel_size=(1, 1), stride=(1, 1))
              (anchor_proj): Conv2d(48, 48, kernel_size=(1, 1), stride=(1, 1))
              (kp_proj): Conv1d(48, 48, kernel_size=(1,), stride=(1,))
              (vp_proj): Conv1d(48, 48, kernel_size=(1,), stride=(1,))
              (out_proj): Conv2d(48, 48, kernel_size=(1, 1), stride=(1, 1))
            )
            (gapl_norm): LayerNorm2d()
            (gapl_ffn): FeedForward(
              (net): Sequential(
                (0): Conv2d(48, 48, kernel_size=(1

## 4. 执行 benchmark 评测

返回结果结构：
- `results[dataset]` 是该数据集逐图结果列表
- 同时写出 `output_csv`


In [9]:
results = evaluate_benchmarks(model, cfg, output_csv=str(output_csv))
print('evaluated datasets =', list(results.keys()))
print('csv saved to =', output_csv)


evaluated datasets = ['Set5', 'Set14', 'BSD100', 'Urban100', 'Manga109']
csv saved to = /Users/dbydd/vibe-agent-working-dir/notes/groupshare/20260514/promptsr-reproduce/outputs/benchmark_results/promptsr_eval.csv


## 5. 汇总逐图结果


In [10]:
rows = []
for dataset_name, dataset_rows in results.items():
    for row in dataset_rows:
        rows.append({
            'dataset': dataset_name,
            'name': row['name'],
            'psnr': row['psnr'],
            'ssim': row['ssim'],
        })

detail_df = pd.DataFrame(rows)
detail_df.sort_values(['dataset', 'name']).reset_index(drop=True)


,dataset,name,psnr,ssim
0,BSD100,101085,15.530788,0.609921
1,BSD100,101087,13.236502,0.644331
2,BSD100,102061,15.509721,0.611331
3,BSD100,103070,17.915469,0.523136
4,BSD100,105025,12.284262,0.432682
...,...,...,...,...
323,Urban100,img_096,14.808386,0.403648
324,Urban100,img_097,14.164072,0.523789
325,Urban100,img_098,15.690670,0.434701
326,Urban100,img_099,14.764106,0.706904


## 6. 查看 benchmark 均值

通常汇报 benchmark 时先看各数据集平均 PSNR / SSIM。


In [11]:
summary_df = (
    detail_df.groupby('dataset', as_index=False)
    .agg(
        num_images=('name', 'count'),
        mean_psnr=('psnr', 'mean'),
        mean_ssim=('ssim', 'mean'),
    )
    .sort_values('dataset')
)
summary_df


,dataset,num_images,mean_psnr,mean_ssim
0,BSD100,100,15.771836,0.525149
1,Manga109,109,12.617260,0.493020
2,Set14,14,14.898420,0.528457
3,Set5,5,15.139879,0.605415
4,Urban100,100,14.388003,0.544430


## 7. 如需读取导出的 CSV


In [12]:
if output_csv.exists():
    pd.read_csv(output_csv).head()
else:
    print('CSV not found yet:', output_csv)
